# Superstore Analytics - SQL Mastery & Ground-Truth Benchmarks
**Project**: Ask Your Data — AI Analytics Copilot  
**Phase**: Phase 4 — SQL Mastery & Verification  
**Database**: `data/superstore.db` (SQLite)

---

## 1. Why SQL Benchmarks Matter
Before deploying an LLM to generate SQL, we must have **ground truth**:
* An exact hand-crafted SQL query for each common business question.
* Verified numerical outputs.
* In Phase 10, we will feed these exact natural language questions to our AI and compare its generated SQL against these benchmarks.


In [1]:
import sqlite3
import pandas as pd

# Connect to our SQLite database in read-only mode
conn = sqlite3.connect("file:../data/superstore.db?mode=ro", uri=True)

# Helper function to run a query and display as a formatted DataFrame
def run_query(sql_query):
    return pd.read_sql_query(sql_query, conn)

print("Connected to data/superstore.db in read-only mode.")


Connected to data/superstore.db in read-only mode.


### Benchmark 1: Overall Totals (Simple Aggregation)
**Question**: What are our overall total sales and total profit across all orders?


In [2]:
sql_1 = """
SELECT 
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore;
"""
run_query(sql_1)


,total_sales,total_profit
0,2297200.86,286397.02


### Benchmark 2: Grouping & Aggregation
**Question**: What are the total sales and profit in each region?


In [3]:
sql_2 = """
SELECT 
    region,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore
GROUP BY region
ORDER BY total_sales DESC;
"""
run_query(sql_2)


,region,total_sales,total_profit
0,West,725457.82,108418.45
1,East,678781.24,91522.78
2,Central,501239.89,39706.36
3,South,391721.91,46749.43


### Benchmark 3: Top Ranking (ORDER BY ... LIMIT)
**Question**: Which top 5 products generated the highest profit?


In [4]:
sql_3 = """
SELECT 
    product_name,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore
GROUP BY product_name
ORDER BY total_profit DESC
LIMIT 5;
"""
run_query(sql_3)


,product_name,total_profit
0,Canon imageCLASS 2200 Advanced Copier,25199.93
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,7753.04
2,Hewlett Packard LaserJet 3310 Copier,6983.88
3,Canon PC1060 Personal Laser Copier,4570.93
4,HP Designjet T520 Inkjet Large Format Printer ...,4094.98


### Benchmark 4: Worst Performers (Unprofitable Outliers)
**Question**: Which 5 products had the largest financial loss?


In [5]:
sql_4 = """
SELECT 
    product_name,
    ROUND(SUM(profit), 2) AS total_loss
FROM superstore
GROUP BY product_name
ORDER BY total_loss ASC
LIMIT 5;
"""
run_query(sql_4)


,product_name,total_loss
0,Cubify CubeX 3D Printer Double Head Print,-8879.97
1,Lexmark MX611dhe Monochrome Laser Printer,-4589.97
2,Cubify CubeX 3D Printer Triple Head Print,-3839.99
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,-2876.12
4,Bush Advantage Collection Racetrack Conference...,-1934.40


### Benchmark 5: Business Ratios & Margins
**Question**: What is the average discount and profit margin percentage by product category?


In [6]:
sql_5 = """
SELECT 
    category,
    ROUND(AVG(discount) * 100, 2) AS avg_discount_pct,
    ROUND(SUM(profit) / SUM(sales) * 100, 2) AS profit_margin_pct
FROM superstore
GROUP BY category
ORDER BY profit_margin_pct DESC;
"""
run_query(sql_5)


,category,avg_discount_pct,profit_margin_pct
0,Technology,13.23,17.40
1,Office Supplies,15.73,17.04
2,Furniture,17.39,2.49


### Benchmark 6: Time Analysis by Year (strftime)
**Question**: What was the total sales and total profit for each year?


In [7]:
sql_6 = """
SELECT 
    strftime('%Y', order_date) AS order_year,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore
GROUP BY order_year
ORDER BY order_year ASC;
"""
run_query(sql_6)


,order_year,total_sales,total_profit
0,2014,484247.50,49543.97
1,2015,470532.51,61618.60
2,2016,609205.60,81795.17
3,2017,733215.26,93439.27


### Benchmark 7: Multi-Condition Filtering (HAVING clause)
**Question**: Which products generated more than $5,000 in sales but had negative profit?


In [8]:
sql_7 = """
SELECT 
    product_name,
    ROUND(SUM(sales), 2) AS total_sales,
    ROUND(SUM(profit), 2) AS total_profit
FROM superstore
GROUP BY product_name
HAVING SUM(sales) > 5000 AND SUM(profit) < 0
ORDER BY total_profit ASC;
"""
run_query(sql_7)


,product_name,total_sales,total_profit
0,Cubify CubeX 3D Printer Double Head Print,11099.96,-8879.97
1,Lexmark MX611dhe Monochrome Laser Printer,16829.90,-4589.97
2,Cubify CubeX 3D Printer Triple Head Print,7999.98,-3839.99
3,Chromcraft Bull-Nose Wood Oval Conference Tabl...,9917.64,-2876.12
4,Bush Advantage Collection Racetrack Conference...,9544.73,-1934.40
5,GBC DocuBind P400 Electric Binding System,17965.07,-1878.17
6,Cisco TelePresence System EX90 Videoconferenci...,22638.48,-1811.08
7,Martin Yale Chadless Opener Electric Letter Op...,16656.20,-1299.18
8,Balt Solid Wood Round Tables,6518.75,-1201.06
9,Hon 2090 “Pillow Soft” Series Mid Back Swivel/...,5282.42,-989.05


### Benchmark 8: Customer & Order Counts (COUNT DISTINCT)
**Question**: How many unique customers and total orders are there in each customer segment?


In [9]:
sql_8 = """
SELECT 
    segment,
    COUNT(DISTINCT customer_id) AS unique_customers,
    COUNT(DISTINCT order_id) AS total_orders,
    ROUND(SUM(sales), 2) AS total_sales
FROM superstore
GROUP BY segment
ORDER BY total_sales DESC;
"""
run_query(sql_8)


,segment,unique_customers,total_orders,total_sales
0,Consumer,409,2586,1161401.34
1,Corporate,236,1514,706146.37
2,Home Office,148,909,429653.15


### Benchmark 9: Monthly Trend Filtering (BETWEEN dates)
**Question**: What was the monthly sales trend in 2017?


In [10]:
sql_9 = """
SELECT 
    strftime('%Y-%m', order_date) AS month,
    ROUND(SUM(sales), 2) AS monthly_sales
FROM superstore
WHERE order_date BETWEEN '2017-01-01' AND '2017-12-31'
GROUP BY month
ORDER BY month ASC;
"""
run_query(sql_9)


,month,monthly_sales
0,2017-01,43971.37
1,2017-02,20301.13
2,2017-03,58872.35
3,2017-04,36521.54
4,2017-05,44261.11
5,2017-06,52981.73
6,2017-07,45264.42
7,2017-08,63120.89
8,2017-09,87866.65
9,2017-10,77776.92


### Benchmark 10: Date Difference Calculation (julianday)
**Question**: What is the average shipping duration in days for each shipping mode?


In [11]:
sql_10 = """
SELECT 
    ship_mode,
    ROUND(AVG(julianday(ship_date) - julianday(order_date)), 2) AS avg_shipping_days
FROM superstore
GROUP BY ship_mode
ORDER BY avg_shipping_days ASC;
"""
run_query(sql_10)


,ship_mode,avg_shipping_days
0,Same Day,0.04
1,First Class,2.18
2,Second Class,3.24
3,Standard Class,5.01
